# Validación empírica — Paper MaxMin Paths\n\nExperimento que verifica dos claims teóricos del paper:\n\n1. **Unicidad de nivel**: con ε > 1/2, cada par de nodos (i,j) aparece exactamente en un orden.\n2. **Escalado de E[η₀]** según el régimen de densidad del grafo:\n   - **Sparse** (grado < 1) → O(1)\n   - **Supercrítico** (1 < grado < D) → Θ(log N)\n   - **Denso** (grado ~ N/2) → O(1)

In [1]:
import sys, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

sys.path.insert(0, '/workspace')
import forgethreads as ft

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
print("Librerías cargadas.")

Librerías cargadas.


## Parámetros globales\nModifica esta celda para cambiar el experimento completo.

In [ ]:
# ── Umbral (epsilon del paper) ────────────────────────────────────────────────
THR       = 0.5    # debe ser > 0.5 para garantizar unicidad de nivel

# ── Orden máximo a explorar ────────────────────────────────────────────────────
MAX_ORDER = 60

# ── Iteraciones por punto (más = más precisión, más tiempo) ───────────────────
N_ITER    = 25

# ── Semilla base para reproducibilidad ────────────────────────────────────────
SEED_BASE = 42

# ── N values para la comparación de regímenes (Claim 2, gráficas 1-3) ─────────
N_VALUES  = [20, 40, 80, 160, 320]

# ── N values para la curva densidad vs E[η₀] (gráfica principal) ──────────────
N_CURVES  = [30, 60, 120, 240]

# ── Grados a barrer en la curva de densidad ────────────────────────────────────
DEG_VALUES = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0, 40.0]

# ── Claim 1: N y grado para la verificación de unicidad ───────────────────────
CLAIM1_N   = 60
CLAIM1_DEGS = [2.0, 3.0, 5.0]

print("Parámetros cargados:"        )
print(f"  THR={THR}  MAX_ORDER={MAX_ORDER}  N_ITER={N_ITER}  SEED_BASE={SEED_BASE}")
print(f"  N_VALUES={N_VALUES}")
print(f"  N_CURVES={N_CURVES}")
print(f"  DEG_VALUES={DEG_VALUES}")

## Generación de grafos\n\nCada arista (i,j) existe con probabilidad `p = avg_degree / (N-1)`.\n- Si existe: peso ~ U(0.55, 1.0) → siempre > ε = 0.5\n- Si no existe: peso = 0.0 → nunca activa caminos con thr = 0.5

In [2]:
def make_graph(N: int, avg_degree: float, rng: np.random.Generator) -> np.ndarray:
    """Grafo aleatorio dirigido [1, N, N] float16."""
    p = min(avg_degree / max(N - 1, 1), 1.0)
    mask = rng.random((N, N)) < p
    np.fill_diagonal(mask, False)
    weights = rng.uniform(0.55, 1.0, (N, N)).astype(np.float16)
    A = np.where(mask, weights, np.float16(0.0)).reshape(1, N, N)
    return A.astype(np.float16)

## Claim 1 — Unicidad de nivel\n\nPara cada par (i,j) con i≠j se registra el **primer orden** en que aparece.\nSe verifican dos invariantes:\n- **Monotonía**: ningún par desaparece entre órdenes consecutivos.\n- **Unicidad**: cada par tiene exactamente un primer orden (trivialmente cierto si hay monotonía).

In [ ]:
def verify_single_level(N=50, avg_degree=3.0, thr=THR, max_order=MAX_ORDER, seed=SEED_BASE):
    rng = np.random.default_rng(seed)
    A = make_graph(N, avg_degree, rng)

    first_order_of = {}
    prev_mn = set()
    pairs_new_per_order = {}
    monotonicity_violations = []

    for order in range(1, max_order + 1):
        paths, values, eff = ft.maxmin(A, A, thr, order)
        p = paths.to_numpy()

        current_mn = set() if p.shape[0] == 0 else {
            (int(row[1]), int(row[-1])) for row in p
        }

        new_pairs = current_mn - prev_mn
        pairs_new_per_order[order] = len(new_pairs)
        for mn in new_pairs:
            first_order_of[mn] = order

        disappeared = prev_mn - current_mn
        if disappeared:
            monotonicity_violations.extend(
                (mn[0], mn[1], first_order_of.get(mn, '?'), order)
                for mn in disappeared
            )

        prev_mn = current_mn
        if order == eff:
            break

    return {
        "ok": len(monotonicity_violations) == 0,
        "pairs_new_per_order": pairs_new_per_order,
        "total_activated": len(first_order_of),
        "violations": monotonicity_violations,
        "effective_order": max(pairs_new_per_order) if pairs_new_per_order else 0,
    }


print(f"Verificando claim 1 (N={CLAIM1_N}, degs={CLAIM1_DEGS}, ε={THR})...")
for deg, seed in zip(CLAIM1_DEGS, [SEED_BASE + i*10 for i in range(len(CLAIM1_DEGS))]):
    r = verify_single_level(N=CLAIM1_N, avg_degree=deg, seed=seed)
    status = "✓ PASS" if r["ok"] else "✗ FAIL"
    print(f"  deg={deg:.1f}  max_order={r['effective_order']}  "
          f"pares={r['total_activated']}  violaciones={len(r['violations'])}  [{status}]")
    print(f"    nuevos por orden: {r['pairs_new_per_order']}")

## Claim 2 — E[η₀] vs N\n\nFunción que estima el orden máximo promedio ejecutando `n_iter` grafos aleatorios.

In [ ]:
def avg_max_order(N, avg_degree, n_iter=N_ITER, thr=THR, max_order=MAX_ORDER, seed=SEED_BASE):
    """Retorna (media, std) del effective_order sobre n_iter grafos aleatorios."""
    rng = np.random.default_rng(seed)
    orders = []
    for _ in range(n_iter):
        A = make_graph(N, avg_degree, rng)
        _, _, eff = ft.maxmin(A, A, thr, max_order)
        orders.append(eff)
    return float(np.mean(orders)), float(np.std(orders))


def run_regime(N_values, deg_fn, label, n_iter=N_ITER, seed=SEED_BASE):
    """Corre el experimento para un régimen y devuelve arrays para graficar."""
    print(f"\n{label}")
    means, stds = [], []
    for N in N_values:
        deg = deg_fn(N)
        m, s = avg_max_order(N, deg, n_iter=n_iter, seed=seed)
        means.append(m); stds.append(s)
        print(f"  N={N:>4}  deg={deg:>6.1f}  E[η₀]={m:.3f} ± {s:.3f}")
    return np.array(means), np.array(stds)

In [ ]:
sparse_m, sparse_s = run_regime(
    N_VALUES, deg_fn=lambda N: 0.7,
    label="SPARSE  (deg=0.7 < 1)", seed=SEED_BASE + 1)

super_m, super_s = run_regime(
    N_VALUES, deg_fn=lambda N: 3.0,
    label="SUPERCRÍTICO  (deg=3.0 > 1)", seed=SEED_BASE + 2)

dense_m, dense_s = run_regime(
    N_VALUES, deg_fn=lambda N: 0.5 * N,
    label="DENSO  (deg=N/2)", seed=SEED_BASE + 3)

## Gráfica 1 — N vs E[η₀] por régimen\n\nEje X: número de nodos N. Eje Y: orden máximo promedio. Banda sombreada: ±1 std.

In [ ]:
Ns = np.array(N_VALUES)

fig, ax = plt.subplots(figsize=(8, 5))

colors_regime = {'sparse': '#2196F3', 'super': '#F44336', 'dense': '#4CAF50'}

for means, stds, label, color in [
    (sparse_m, sparse_s, 'Sparse  (deg=0.7, O(1))',        colors_regime['sparse']),
    (super_m,  super_s,  'Supercrítico  (deg=3, Θ(log N))', colors_regime['super']),
    (dense_m,  dense_s,  'Denso  (deg=N/2, O(1))',          colors_regime['dense']),
]:
    ax.plot(Ns, means, 'o-', color=color, label=label, linewidth=2, markersize=6)
    ax.fill_between(Ns, means - stds, means + stds, color=color, alpha=0.15)

ax.set_xlabel('N  (número de nodos)', fontsize=12)
ax.set_ylabel('E[η₀]  (orden máximo promedio)', fontsize=12)
ax.set_title('Orden máximo promedio vs tamaño del grafo', fontsize=13)
ax.legend(fontsize=10)
ax.set_xticks(Ns)
ax.set_xticklabels(Ns)

plt.tight_layout()
plt.savefig('results_N_vs_order.png', dpi=150, bbox_inches='tight')
plt.show()

## Gráfica 2 — Supercrítico vs log(N)\n\nSi E[η₀] ~ c·log(N), la curva en escala log-X debe ser una recta.\nEl ratio E[η₀]/log(N) debe converger a una constante.

In [ ]:
logNs = np.log(Ns)
ratios = super_m / logNs

# Ajuste lineal E[η₀] = a·log(N) + b
a, b = np.polyfit(logNs, super_m, 1)
fit_line = a * logNs + b

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Izquierda: E[η₀] vs log(N) con ajuste lineal ─────────────────────────
ax = axes[0]
ax.plot(logNs, super_m, 'o', color=colors_regime['super'], markersize=7, label='E[η₀] supercrítico')
ax.fill_between(logNs, super_m - super_s, super_m + super_s,
                color=colors_regime['super'], alpha=0.15)
ax.plot(logNs, fit_line, '--', color='gray',
        label=f'Ajuste: {a:.2f}·log(N) + {b:.2f}')
ax.set_xlabel('log(N)', fontsize=12)
ax.set_ylabel('E[η₀]', fontsize=12)
ax.set_title('Supercrítico: E[η₀] vs log(N)', fontsize=12)
ax.legend(fontsize=10)

# ── Derecha: ratio E[η₀] / log(N) ────────────────────────────────────────
ax = axes[1]
ax.plot(Ns, ratios, 's-', color=colors_regime['super'], linewidth=2, markersize=6)
ax.axhline(y=np.mean(ratios[-3:]), color='gray', linestyle='--',
           label=f'Asíntota ≈ {np.mean(ratios[-3:]):.3f}')
ax.set_xlabel('N', fontsize=12)
ax.set_ylabel('E[η₀] / log(N)', fontsize=12)
ax.set_title('Ratio E[η₀]/log(N)  (debe → constante)', fontsize=12)
ax.set_xticks(Ns)
ax.set_xticklabels(Ns)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('results_logN_fit.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAjuste lineal: E[η₀] ≈ {a:.3f}·log(N) + {b:.3f}")
print(f"Ratio E[η₀]/log(N) para los últimos 3 puntos: {ratios[-3:]}")
print(f"Valor asintótico estimado: {np.mean(ratios[-3:]):.4f}")

## Gráfica 3 — Tabla resumen y comparación de los tres regímenes

In [ ]:
print(f"{'N':>5}  {'log N':>6}  {'sparse':>8}  {'supercrit':>10}  {'dense':>7}  {'ratio sc/logN':>14}")
print("─" * 62)
for i, N in enumerate(N_VALUES):
    ratio = super_m[i] / math.log(N)
    print(f"{N:>5}  {math.log(N):>6.2f}  {sparse_m[i]:>8.3f}  "
          f"{super_m[i]:>10.3f}  {dense_m[i]:>7.3f}  {ratio:>14.4f}")

# Gráfica comparativa con escala log en X
fig, ax = plt.subplots(figsize=(9, 5))

for means, stds, label, color, marker in [
    (sparse_m, sparse_s, 'Sparse  (deg=0.7)',   colors_regime['sparse'], 'o'),
    (super_m,  super_s,  'Supercrítico (deg=3)', colors_regime['super'],  's'),
    (dense_m,  dense_s,  'Denso  (deg=N/2)',     colors_regime['dense'],  '^'),
]:
    ax.semilogx(Ns, means, f'{marker}-', color=color, label=label,
                linewidth=2, markersize=7)
    ax.fill_between(Ns, means - stds, means + stds, color=color, alpha=0.15)

# Referencia log(N)
log_ref = np.log(Ns)
ax.semilogx(Ns, log_ref / log_ref[-1] * super_m[-1], ':k',
            alpha=0.4, linewidth=1.5, label='Referencia log(N)')

ax.set_xlabel('N  (escala log)', fontsize=12)
ax.set_ylabel('E[η₀]', fontsize=12)
ax.set_title('Comparación de regímenes: E[η₀] vs N  (eje X en log)', fontsize=13)
ax.set_xticks(Ns)
ax.set_xticklabels(Ns)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Gráfica principal — Densidad vs E[η₀]\n\nEje X: grado promedio (de sparse → supercrítico → denso).  \nEje Y: orden máximo promedio E[η₀].  \nUna curva por cada tamaño de grafo N.

In [ ]:
results = {}   # results[N] = (deg_vals_used, means, stds)

for N in N_CURVES:
    print(f"N={N} ...", end=' ', flush=True)
    degs = [d for d in DEG_VALUES if d < N - 1]
    ms, ss = [], []
    for deg in degs:
        m, s = avg_max_order(N, deg, seed=N * 100)
        ms.append(m)
        ss.append(s)
    results[N] = (degs, np.array(ms), np.array(ss))
    print("done")

print("Experimento completado.")

In [ ]:
cmap   = plt.cm.plasma
colors = [cmap(i / (len(N_CURVES) - 1)) for i in range(len(N_CURVES))]

fig, ax = plt.subplots(figsize=(10, 6))

for color, N in zip(colors, N_CURVES):
    degs, ms, ss = results[N]
    ax.plot(degs, ms, 'o-', color=color, label=f'N = {N}',
            linewidth=2, markersize=6)
    ax.fill_between(degs, ms - ss, ms + ss, color=color, alpha=0.12)

# Líneas de referencia para los tres regímenes
ax.axvline(x=1.0,  color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(x=10.0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(0.55,  ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 1 else 5,
        'sparse\n(deg<1)', fontsize=9, color='gray', ha='center')
ax.text(3.5,   ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 1 else 5,
        'supercrítico', fontsize=9, color='gray', ha='center')
ax.text(20.0,  ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 1 else 5,
        'denso', fontsize=9, color='gray', ha='center')

ax.set_xscale('log')
ax.set_xlabel('Grado promedio (escala log)', fontsize=13)
ax.set_ylabel('E[η₀]  —  orden máximo promedio', fontsize=13)
ax.set_title('Orden máximo promedio vs densidad del grafo', fontsize=14)
ax.legend(title='Nodos N', fontsize=11, title_fontsize=11)

ax.set_xticks(DEG_VALUES)
ax.set_xticklabels([str(d) for d in DEG_VALUES], rotation=30, fontsize=9)
ax.xaxis.set_minor_formatter(ticker.NullFormatter())

plt.tight_layout()
plt.savefig('results_density_curve.png', dpi=150, bbox_inches='tight')
plt.show()